# NANO-GPT 150M BPE — этап «связные предложения»
BPE-токены (контекст 512 токенов ~= 2500 символов) + корпус 1.2 ГБ (русская Википедия)
+ наши диалоги x2 для чат-навыка. Тёплый старт из старой char-модели: трансформер уже умеет
складывать слова, переучиваются только эмбеддинги токенов.

## Ротация 7 аккаунтов
Чекпоинты и корпус лежат в GitHub-релизе репо — общий для всех аккаунтов.
• <b>Аккаунт #1 (один раз):</b> ячейки 1→2→3→5(сборка)→6(выгрузка данных)→7(обучение)
• <b>Крэш на том же аккаунте:</b> 1→2→3→8(резюм с Драйва)
• <b>Смена аккаунта (вечер):</b> на старом 9(выгрузка чекпоинта), на новом 1→2→3→4(скачать всё)→7
Не запускай 5-6 на аккаунтах #2+ — корпус уже собран.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
if not os.path.exists('/content/nano-gpt'):
    !git clone -q https://github.com/lucifermornngstar52-cell/nano-gpt.git /content/nano-gpt
%cd /content/nano-gpt
!pip -q install datasets tokenizers

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# [АККАУНТЫ #2+] Скачать корпус, токенизатор и чекпоинт с релиза (~10-15 мин)
import requests, os
base = 'https://github.com/lucifermornngstar52-cell/nano-gpt/releases/download/bpe/'
for fn, mb in [('bpe_tokenizer.json', 0.3), ('tokens.bin', 700), ('weights_bpe_fp16.npz', 300),
               ('step.txt', 0), ('meta_bpe.json', 0)]:
    if os.path.exists(fn):
        print(fn, 'уже есть'); continue
    r = requests.get(base + fn, stream=True, timeout=600)
    if r.status_code != 200:
        print(fn, 'нет в релизе (ок для первого аккаунта)'); continue
    with open(fn, 'wb') as f:
        for chunk in r.iter_content(1 << 20):
            f.write(chunk)
    print(fn, '%.0f МБ' % (os.path.getsize(fn)/1e6))
import os
if os.path.exists('weights_bpe_fp16.npz'):
    os.replace('weights_bpe_fp16.npz', 'weights_bpe.npz')
    print('чекпоинт готов к резюму с шага', open('step.txt').read().strip() if os.path.exists('step.txt') else '?')

In [ ]:
# [ТОЛЬКО АККАУНТ #1] Сборка: вики 1.2 ГБ + диалоги x2 -> BPE -> tokens.bin (~1-1.5 ч)
import os
os.environ.update(NBYTES='1200000000', VOCAB='8192', WIKI='1',
                  DIAL='corpus_big.txt.gz', OUT='corpus_mix.txt.gz')
%cd /content/nano-gpt
!python bpe_build.py

In [ ]:
# [ТОЛЬКО АККАУНТ #1] Выложить tokens.bin + токенизатор в релиз (чтобы остальные не пересобирали)
import requests, subprocess, re, os, numpy as np
url = subprocess.check_output(['git', 'remote', 'get-url', 'origin'],
        cwd='/content/nano-gpt').decode()
tok = re.search(r':(ghp_[^@]+)@', url).group(1)
repo = 'lucifermornngstar52-cell/nano-gpt'
H = {'Authorization': 'token ' + tok}
r = requests.get(f'https://api.github.com/repos/{repo}/releases/tags/bpe', headers=H).json()
if 'id' not in r:
    r = requests.post(f'https://api.github.com/repos/{repo}/releases', headers=H,
        json={'tag_name': 'bpe', 'name': 'BPE corpus & checkpoints'}).json()
up, rid = r['upload_url'].split('{')[0], r['id']
for old in r.get('assets', []):
    if old['name'] in ('tokens.bin', 'bpe_tokenizer.json'):
        requests.delete(f"https://api.github.com/repos/{repo}/releases/assets/{old['id']}", headers=H)
for fn in ('bpe_tokenizer.json', 'tokens.bin'):
    if os.path.exists(fn):
        with open(fn, 'rb') as f:
            resp = requests.post(up, params={'name': fn}, headers=H,
                                 data=f.read(), timeout=3600)
        print(fn, '->', resp.status_code)

In [ ]:
# ОБУЧЕНИЕ (~7-9 ч ночи на T4). WARM=1 только если рядом старая char-модель.
import os, shutil
os.makedirs('/content/drive/MyDrive/nano-gpt-bpe', exist_ok=True)
warm = '0'
src = '/content/drive/MyDrive/nano-gpt-150m/weights_big.npz'
if os.path.exists(src) and not os.path.exists('weights_bpe.npz'):
    shutil.copy(src, 'weights_src.npz'); warm = '1'
    print('тёплый старт из char-модели')
os.environ.update(DM='1024', NL='12', NH='16', CTX='512', BS='24', LR='6e-4',
                  STEPS='40000', TOKENS='tokens.bin', TOK='bpe_tokenizer.json',
                  CKPT='/content/drive/MyDrive/nano-gpt-bpe', WARM=warm)
%cd /content/nano-gpt
!python train_bpe.py

In [ ]:
# РЕЗЮМ на ЭТОМ аккаунте (после обрыва Colab)
import os
from google.colab import drive
drive.mount('/content/drive')
%cd /content/nano-gpt
ck = '/content/drive/MyDrive/nano-gpt-bpe'
for f in ('weights_bpe.npz', 'meta_bpe.json', 'step.txt', 'bpe_tokenizer.json'):
    os.system(f'cp -f {ck}/{f} . 2>/dev/null')
os.environ.update(DM='1024', NL='12', NH='16', CTX='512', BS='24', LR='6e-4',
                  STEPS='40000', TOKENS='tokens.bin', TOK='bpe_tokenizer.json',
                  CKPT=ck, WARM='0')
print('резюм с шага', open('step.txt').read() if os.path.exists('step.txt') else '?')
!python train_bpe.py

In [ ]:
# СМЕНА АККАУНТА (вечером): выложить свежий чекпоинт в релиз (~5 мин)
import requests, subprocess, re, os, numpy as np
z = np.load('weights_bpe.npz')
z16 = {k: np.asarray(a, dtype=np.float16) for k, a in z.items()}
np.savez('weights_bpe_fp16.npz', **z16)
url = subprocess.check_output(['git', 'remote', 'get-url', 'origin'],
        cwd='/content/nano-gpt').decode()
tok = re.search(r':(ghp_[^@]+)@', url).group(1)
repo = 'lucifermornngstar52-cell/nano-gpt'
H = {'Authorization': 'token ' + tok}
r = requests.get(f'https://api.github.com/repos/{repo}/releases/tags/bpe', headers=H).json()
up, rid = r['upload_url'].split('{')[0], r['id']
names = ('weights_bpe_fp16.npz', 'step.txt', 'meta_bpe.json')
for old in r.get('assets', []):
    if old['name'] in names:
        requests.delete(f"https://api.github.com/repos/{repo}/releases/assets/{old['id']}", headers=H)
for fn in names:
    if os.path.exists(fn):
        with open(fn, 'rb') as f:
            resp = requests.post(up, params={'name': fn}, headers=H, data=f.read(), timeout=1800)
        print(fn, '->', resp.status_code)
print('теперь на другом аккаунте: ячейки 1,2,3,4,7')

In [ ]:
# Чат-тест свежей модели
import os, json, torch, numpy as np
os.environ.update(DM='1024', NL='12', NH='16', CTX='512')
%cd /content/nano-gpt
from tokenizers import Tokenizer
tk = Tokenizer.from_file('bpe_tokenizer.json')
import train_bpe as tb  # поднимет своё, но нам нужен только класс
V = tk.get_vocab_size()
model = tb.NanoT().to(tb.dev)
z = np.load('weights_bpe.npz')
with torch.no_grad():
    model.we.copy_(torch.tensor(z['we']).float()); model.wpe.copy_(torch.tensor(z['wpe']).float())
    for l in range(tb.NL):
        b = model.blocks[l]
        for nm in ('q','k','v','o','fc','fc2'):
            getattr(b, nm).weight.copy_(torch.tensor(z[f'{l}{nm}']).float())
        b['ln1'].weight.copy_(torch.tensor(z[f'{l}ln1g'])); b['ln1'].bias.copy_(torch.tensor(z[f'{l}ln1b']))
        b['ln2'].weight.copy_(torch.tensor(z[f'{l}ln2g'])); b['ln2'].bias.copy_(torch.tensor(z[f'{l}ln2b']))
    model.lnf.weight.copy_(torch.tensor(z['lnfg'])); model.lnf.bias.copy_(torch.tensor(z['lnfb']))
model.eval()
@torch.no_grad()
def reply(q, temp=0.8, n=80):
    ids = tk.encode('В: ' + q + '\nО:').ids
    out = []
    for _ in range(n):
        lg = model(torch.tensor([ids[-512:]], device=tb.dev))[0, -1] / temp
        k = torch.topk(lg, 8)
        ni = k.indices[torch.multinomial(torch.softmax(k.values, -1), 1)].item()
        ids.append(ni); out.append(ni)
        if tk.decode(out[-3:]).endswith('\n\n'):
            out = out[:-3]; break
    return tk.decode(out).strip()
for q in ['привет', 'как дела?', 'расскажи о космосе']:
    print('В:', q, '\nО:', reply(q), '\n')